# DeepSeek OCR Testing

In [2]:
from transformers import AutoModel, AutoTokenizer
import torch
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

model_name = "deepseek-ai/DeepSeek-OCR-2"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
    _attn_implementation="flash_attention_2",
).eval().cuda()
print("Model and tokenizer loaded.")

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.


Model and tokenizer loaded.


In [ ]:
from contextlib import redirect_stdout
from io import StringIO
import json
import re
from pathlib import Path

import fitz

ROOT = Path.cwd()
NAMESPACE = "exp_mbg_v1"
pdf_path = ROOT / "data" / "raw" / "pdf" / "luxembourgish_grammar.pdf"
output_dir = ROOT / "artifacts" / "experiments" / NAMESPACE / "ocr" / "deepseek_page1"
output_dir.mkdir(parents=True, exist_ok=True)

page_number = 1
page_image_path = output_dir / f"page_{page_number:04d}.png"
page = fitz.open(pdf_path).load_page(page_number - 1)
zoom = 220 / 72.0
pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
pix.save(page_image_path)

prompt = "<image>\n<|grounding|>Convert the document to markdown. "
ocr_stdout = StringIO()
with redirect_stdout(ocr_stdout):
    res = model.infer(
        tokenizer,
        prompt=prompt,
        image_file=str(page_image_path),
        output_path=str(output_dir),
        base_size=1024,
        image_size=768,
        crop_mode=True,
        save_results=True,
    )

raw_stdout = ocr_stdout.getvalue()
result_md_path = output_dir / "result.mmd"
markdown_text = result_md_path.read_text(encoding="utf-8") if result_md_path.exists() else ""

marker_pattern = re.compile(
    r"^<\|ref\|>(?P<label>.*?)<\|/ref\|><\|det\|>(?P<bbox>\[\[.*?\]\])<\|/det\|>$"
)
blocks = []
current_block = None
for line in raw_stdout.splitlines():
    marker_match = marker_pattern.match(line.strip())
    if marker_match:
        if current_block is not None:
            current_block["text"] = "\n".join(current_block["text_lines"]).strip()
            current_block.pop("text_lines", None)
            blocks.append(current_block)
        current_block = {
            "label": marker_match.group("label"),
            "bbox": json.loads(marker_match.group("bbox"))[0],
            "text_lines": [],
        }
        continue
    if current_block is not None:
        if line.startswith("===============save results:="):
            current_block["text"] = "\n".join(current_block["text_lines"]).strip()
            current_block.pop("text_lines", None)
            blocks.append(current_block)
            current_block = None
            break
        current_block["text_lines"].append(line)

if current_block is not None:
    current_block["text"] = "\n".join(current_block["text_lines"]).strip()
    current_block.pop("text_lines", None)
    blocks.append(current_block)

structured_result = {
    "pdf_path": str(pdf_path),
    "page_number": page_number,
    "page_image_path": str(page_image_path),
    "output_dir": str(output_dir),
    "result_markdown_path": str(result_md_path),
    "result_markdown": markdown_text,
    "stdout_blocks": blocks,
    "raw_stdout": raw_stdout,
}

structured_json_path = output_dir / f"page_{page_number:04d}_ocr_structured.json"
structured_json_path.write_text(json.dumps(structured_result, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Rendered page image: {page_image_path}")
print(f"OCR markdown: {result_md_path}")
print(f"Structured JSON: {structured_json_path}")
print(f"Captured blocks: {len(blocks)}")
print(f"Markdown chars: {len(markdown_text)}")
print("\n--- Markdown preview ---\n")
print(markdown_text[:2000])


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
image: 0it [00:00, ?it/s]
other: 100%|██████████| 4/4 [00:00<00:00, 79137.81it/s]

Rendered page image: /home/snt/projects/GramMetaRL/artifacts/deepseek_ocr_page1/page_0001.png
OCR markdown: /home/snt/projects/GramMetaRL/artifacts/deepseek_ocr_page1/result.mmd
Structured JSON: /home/snt/projects/GramMetaRL/artifacts/deepseek_ocr_page1/page_0001_ocr_structured.json
Captured blocks: 4
Markdown chars: 1466

--- Markdown preview ---


## Luxembourgish


## Summary


This article provides an overview of the structure of the Luxembourgish language, the national language of the Grand Duchy of Luxembourg, which has developed from a Moselle Franconian dialect to an Ausbau language in the course of the 20th century. In the early 21st century, Luxembourgish serves several functions, mainly as a multifunctional spoken variety but also as a written language, which has acquired a medium level of language standardization. Because of the embedding into a complex multilingual situation with German and French, Luxembourgish is characterized by a high degree of language contact. As a G